# allelign in a notebook: download, read, align

The workflow of [walkthrough.md](walkthrough.md) as Python: resolve and
download a published file, read it, align it to a reference, and record
what each stage kept or dropped. The one network step downloads a PGS
Catalog score of 77 variants (a few KB); everything else runs on what
is already on disk. The rules behind every call are in
[guide.md](guide.md).


In [ ]:
import numpy as np
import allelign as al

al.__version__


## 1. Resolve and download

`resolve_ref` names one published file: it HEAD-checks the URL and
returns the plan a `--dry-run` prints on the command line. `PGS000001`
is a classic 77-variant breast-cancer score, published in the
submitted layout.


In [ ]:
plan = al.resolve_ref("PGS000001")
{key: plan[key] for key in
 ("provider", "label", "genome_build", "trait", "remote_bytes")}


In [ ]:
record = al.download_ref("PGS000001", "PGS000001.txt.gz")
record


The provenance sidecar `<out>.meta.json` holds the same record — keep it
with the data.


## 2. Read the score

`read_scoring_file` applies the PGS Catalog rules (guide.md §4):
harmonised columns preferred when published, OR/HR weights logged,
non-additive rows dropped, and the coordinate build read from the
columns themselves.


In [ ]:
sf = al.read_scoring_file("PGS000001.txt.gz")
print(len(sf), "variants; build", sf.genome_build)
for row in zip(sf.id[:5], sf.chrom[:5], sf.pos[:5], sf.ea[:5],
               sf.oa[:5], sf.weight[:5]):
    print(row)


## 3. Align the score to a reference

Build a small reference panel from the score's own variants (a real run
uses `read_bim`/`read_pvar`). Swapping `a1`/`a2` on the first 25
reference variants makes the orientation work visible. Scores that
identify variants by position alone — PGS000337's harmonised file, for
example, leaves `rsID` and the other allele empty — match by
coordinates instead; a real reference panel supplies both.


In [ ]:
ea = np.asarray(sf.ea, dtype=object)
oa = np.asarray(sf.oa, dtype=object)
a1, a2 = ea.copy(), oa.copy()
a1[:25] = oa[:25].copy()    # counted allele swapped on the first 25
a2[:25] = ea[:25].copy()
ref = al.Variants(
    chrom=np.asarray(sf.chrom, dtype=object),
    id=np.asarray(sf.id, dtype=object),
    pos=np.asarray(sf.pos, dtype=np.int64),
    a1=a1, a2=a2).freeze()
len(ref)


In [ ]:
w = al.harmonize_weights(sf, ref)
print(w.log)
print("flipped:", int(w.flipped.sum()), "of", len(w))


A flip means the effect allele was the reference's other allele, so the
weight changed sign; `weight_mass_matched` is the share of the total
squared weight that reached the reference.


In [ ]:
i = int(np.flatnonzero(w.flipped)[0])
print("before:", float(sf.weight[w.src_index[i]]))
print("after: ", float(w.weight[i]))


## 4. A miniature sumstats pipeline

Treat the score as statistics: write a sumstats file, read it back,
QC it with a per-row status record, and harmonise it against the same
reference. This is the `clean` + `align` flow of the CLI, one call per
stage.


In [ ]:
rng = np.random.default_rng(7)
ss = al.Sumstats(
    id=np.asarray(sf.id, dtype=object),
    chrom=np.asarray(sf.chrom, dtype=object),
    pos=np.asarray(sf.pos, dtype=np.int64),
    ea=np.asarray(sf.ea, dtype=object),
    oa=np.asarray(sf.oa, dtype=object),
    beta=np.asarray(sf.weight, dtype=float) / 10.0,
    se=rng.uniform(0.01, 0.05, len(sf)),
    n_eff=np.full(len(sf), 100000.0),
    eaf=np.full(len(sf), np.nan), info=np.full(len(sf), np.nan))
ss.write("mini.tsv.gz")
mini = al.read_sumstats("mini.tsv.gz", n_eff=100000)
len(mini)


In [ ]:
status = al.new_status(len(mini))
keep, qc_log = al.qc_sumstats(mini, status=status)
qc_log, al.status_counts(status)


Any dropped row is remembered by the stage that dropped it — the status
vocabulary is the CLI's `--status-out` table.


In [ ]:
sub = mini.subset(keep)
h = al.harmonize(sub, ref)
coverage = al.reference_coverage(sub, ref, h)
al.coverage_message(coverage, label="mini") or "coverage ok"


In [ ]:
al.write_status_table("mini.status.tsv", id=mini.id, chrom=mini.chrom,
                        pos=mini.pos, ea=mini.ea, oa=mini.oa,
                        codes=status)
print("\n".join(open("mini.status.tsv").read().splitlines()[:3]))


## 5. Where to go next

- [guide.md](guide.md) — every rule, default and deliberate difference
  from ldpred3, plus the source-reference table (§6).
- [walkthrough.md](walkthrough.md) — the same flow through the CLI,
  including a full study download restricted to a reference's variants.
